In [1]:
"""
NER Examples in Python
Author: Phd Abdelouaheb

This file demonstrates several NER workflows:
1) Quick inference with spaCy
2) EntityRuler (rule-based patterns) for domain terms
3) Hugging Face Transformers pipeline
4) Simple span-level evaluation (precision/recall/F1)
5) Regex + gazetteer bootstrap
6) Converting entities to BIO tags (utility)
NOTE: Installation commands are commented out. Uncomment in your own environment if needed.
"""

# --- Setup (uncomment as needed) ---
# pip install spacy transformers torch --upgrade
# python -m spacy download en_core_web_sm

from typing import List, Tuple, Dict
from collections import Counter
import re

# ============== 1) spaCy: quick NER ==============
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    text = "Apple acquired Beats for $3 billion in 2014 in Cupertino, California. Tim Cook commented."
    doc = nlp(text)
    print("\n[spaCy] Entities:")
    for ent in doc.ents:
        print(f"{ent.text:25s}  {ent.label_:10s}")
except Exception as e:
    print("[spaCy] Not available or model missing:", e)

# ============== 2) spaCy: EntityRuler patterns (domain boosting) ==============
try:
    import spacy
    from spacy.pipeline import EntityRuler

    # Start from a blank or an existing model
    try:
        nlp2 = spacy.load("en_core_web_sm")
    except Exception:
        nlp2 = spacy.blank("en")

    ruler = nlp2.add_pipe("entity_ruler", config={"overwrite_ents": True})
    patterns = [
        {"label": "PRODUCT", "pattern": "iPhone 15 Pro"},
        {"label": "ORG", "pattern": [{"LOWER": "csa"}, {"LOWER": "-", "OP": "?"}, {"LOWER": "research"}]},
        {"label": "LAW", "pattern": "GDPR"},
        {"label": "EVENT", "pattern": "WWDC 2024"},
    ]
    ruler.add_patterns(patterns)

    doc2 = nlp2("CSA Research analyzed iPhone 15 Pro adoption under GDPR. See you at WWDC 2024!")
    print("\n[spaCy + EntityRuler] Entities:")
    for ent in doc2.ents:
        print(f"{ent.text:25s}  {ent.label_:10s}")
except Exception as e:
    print("[EntityRuler] Could not run:", e)

# ============== 3) Hugging Face: token-classification pipeline ==============
try:
    from transformers import pipeline
    # Uses a small default English model; replace with a domain model from HF Hub as needed.
    ner = pipeline("token-classification", aggregation_strategy="simple")
    text3 = "OpenAI is headquartered in San Francisco and partners with Microsoft."
    print("\n[HF pipeline] Entities:")
    for ent in ner(text3):
        print(ent)
except Exception as e:
    print("[HF] Transformers pipeline not available:", e)

# ============== 4) Span-level evaluation ==============
Span = Tuple[int, int, str]  # (start_char, end_char, label)

def extract_spans_spacy(doc) -> List[Span]:
    return [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]

def span_prf1(pred: List[Span], gold: List[Span]) -> Dict[str, float]:
    pred_set = set(pred)
    gold_set = set(gold)
    tp = len(pred_set & gold_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return {"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4)}

# Tiny demo with one sentence (for illustration only)
try:
    import spacy
    nlp_eval = spacy.load("en_core_web_sm")
    sent = "Amazon Web Services operates in Virginia; revenue reached $25 billion in Q2 2024."
    gold_spans: List[Span] = [
        (0, 20, "ORG"),          # Amazon Web Services
        (34, 42, "GPE"),         # Virginia
        (63, 74, "MONEY"),       # $25 billion
        (78, 85, "DATE"),        # Q2 2024  (spaCy may tag as DATE)
    ]
    pred_spans = extract_spans_spacy(nlp_eval(sent))
    metrics = span_prf1(pred_spans, gold_spans)
    print("\n[Eval] predicted spans:", pred_spans)
    print("[Eval] gold spans     :", gold_spans)
    print("[Eval] metrics        :", metrics)
except Exception as e:
    print("[Eval] Could not run evaluation demo:", e)

# ============== 5) Regex + gazetteer bootstrap ==============
GAZETTEER_ORGS = {"OpenAI", "CSA Research", "Microsoft", "Amazon", "Apple"}

def regex_money(text: str) -> List[Span]:
    spans = []
    for m in re.finditer(r"\$?\s?\d+(?:\.\d+)?\s?(?:billion|million|k|B|M|K)?", text, re.IGNORECASE):
        spans.append((m.start(), m.end(), "MONEY"))
    return spans

def gazetteer_orgs(text: str) -> List[Span]:
    spans = []
    for org in sorted(GAZETTEER_ORGS, key=len, reverse=True):
        for m in re.finditer(re.escape(org), text):
            spans.append((m.start(), m.end(), "ORG"))
    return spans

def merge_spans(*lists: List[Span]) -> List[Span]:
    # naive merge (keeps all; de-duplicate exact matches)
    merged = set()
    for lst in lists:
        for s in lst:
            merged.add(s)
    return sorted(list(merged))

text4 = "OpenAI raised $10 billion with Microsoft; CSA Research estimated $3.5M in 2025."
boot_spans = merge_spans(regex_money(text4), gazetteer_orgs(text4))
print("\n[Bootstrap] spans:", boot_spans)

# ============== 6) Convert character spans to BIO tags ==============
def to_bio(tokens: List[str], token_offsets: List[Tuple[int, int]], spans: List[Span]) -> List[str]:
    """Assign BIO tags to tokens given character-level spans."""
    tags = ["O"] * len(tokens)
    # index spans by character ranges
    for (s, e, label) in spans:
        for i, (ts, te) in enumerate(token_offsets):
            if ts >= s and te <= e:
                prefix = "B" if ts == s else "I"
                tags[i] = f"{prefix}-{label}"
    return tags

# Simple whitespace tokenizer with char offsets
def simple_tokenize_with_offsets(text: str) -> Tuple[List[str], List[Tuple[int,int]]]:
    tokens, offsets = [], []
    idx = 0
    for part in re.finditer(r"\S+", text):
        s, e = part.start(), part.end()
        tokens.append(text[s:e])
        offsets.append((s, e))
    return tokens, offsets

tokens5, offs5 = simple_tokenize_with_offsets(text4)
bio_tags = to_bio(tokens5, offs5, boot_spans)
print("\n[Utility] Tokens + BIO:")
for t, tag in zip(tokens5, bio_tags):
    print(f"{t:15s} {tag}")

print("\nDone.")



[spaCy] Entities:
Apple                      ORG       
$3 billion                 MONEY     
2014                       DATE      
Cupertino                  GPE       
California                 GPE       
Tim Cook                   PERSON    

[spaCy + EntityRuler] Entities:
CSA Research               ORG       
iPhone 15 Pro              PRODUCT   
GDPR                       LAW       
WWDC 2024                  EVENT     


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

C:\Users\ASUS TUF\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS TUF\.cache\huggingface\hub\models--dbmdz--bert-large-cased-finetuned-conll03-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]


[HF pipeline] Entities:
{'entity_group': 'ORG', 'score': 0.9978426, 'word': 'OpenAI', 'start': 0, 'end': 6}
{'entity_group': 'LOC', 'score': 0.9985428, 'word': 'San Francisco', 'start': 27, 'end': 40}
{'entity_group': 'ORG', 'score': 0.99959594, 'word': 'Microsoft', 'start': 59, 'end': 68}

[Eval] predicted spans: [(0, 19, 'ORG'), (32, 40, 'GPE'), (58, 69, 'MONEY'), (73, 80, 'DATE')]
[Eval] gold spans     : [(0, 20, 'ORG'), (34, 42, 'GPE'), (63, 74, 'MONEY'), (78, 85, 'DATE')]
[Eval] metrics        : {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

[Bootstrap] spans: [(0, 6, 'ORG'), (14, 25, 'MONEY'), (31, 40, 'ORG'), (42, 54, 'ORG'), (65, 70, 'MONEY'), (73, 78, 'MONEY')]

[Utility] Tokens + BIO:
OpenAI          B-ORG
raised          O
$10             B-MONEY
billion         I-MONEY
with            O
Microsoft;      O
CSA             B-ORG
Research        I-ORG
estimated       O
$3.5M           B-MONEY
in              O
2025.           O

Done.
